<a href="https://colab.research.google.com/github/azaher1215/CSE676-Capstone/blob/main/Itercomp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q datasets nltk FlagEmbedding openai transformers accelerate bitsandbytes

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Data Loader

In [ ]:
import json, random, urllib.request, re, string, collections, time
import numpy as np
from datasets import load_dataset

In [ ]:
# Load datasets

def load_hotpotqa():
    return load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation")

def load_2wiki():
    return load_dataset("framolfese/2WikiMultihopQA", split="validation")

def load_musique():
    return load_dataset("dgslibisey/MuSiQue", split="validation")

In [ ]:
# Parse into a common shape

def parse_hotpot_style(ex):
    ctx, sf = ex["context"], ex["supporting_facts"]
    if isinstance(ctx, dict):                       # dict-of-lists (HF)
        gold = set(sf["title"])
        items = zip(ctx["title"], ctx["sentences"])
    else:                                           # list-of-pairs
        gold = {title for title, _ in sf}
        items = ctx
    paragraphs = [
        {"title": title, "text": " ".join(sentences), "is_supporting": title in gold}
        for title, sentences in items
    ]
    return {"question": ex["question"], "answer": ex["answer"], "paragraphs": paragraphs}


def parse_musique(ex):
    paragraphs = [
        {"title": p["title"], "text": p["paragraph_text"], "is_supporting": p["is_supporting"]}
        for p in ex["paragraphs"]
    ]
    return {"question": ex["question"], "answer": ex["answer"],
            "answer_aliases": ex.get("answer_aliases", []) or [],   # MuSiQue accepts these
            "paragraphs": paragraphs}


In [ ]:
# Pick samples
def sample(data, n=20, seed=42):
    random.seed(seed)
    return random.sample(list(data), n)

In [ ]:
# get one dataset at a time
def get(dataset, n=20, seed=42):
    if dataset == "hotpotqa":
        raw, parse = load_hotpotqa(), parse_hotpot_style
    elif dataset == "2wiki":
        raw, parse = load_2wiki(), parse_hotpot_style
    elif dataset == "musique":
        raw, parse = load_musique(), parse_musique
    return [parse(ex) for ex in sample(raw, n, seed)]

#Document Decomposition

In [ ]:
# Decompose using sentence level
import nltk
nltk.download("punkt_tab", quiet=True)
from nltk.tokenize import sent_tokenize

def decompose(record):
  segments = []
  for p in record["paragraphs"]:
    for sentence in sent_tokenize(p["text"]):
      segments.append({
          "text":sentence,  # the sentence itself
          "title":p["title"], # which paragraph it came from
          "is_supporting":p["is_supporting"] # gold vs distractor
      })
  return segments

# Dual-Aspect Relevance Filtering

In [ ]:
_model = None
def get_model():
    global _model
    if _model is None:
        from FlagEmbedding import BGEM3FlagModel
        _model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)
    return _model
def score_and_filter(question, segments, lam=0.6, k=90):
    model = get_model()
    seg_texts = [s["text"] for s in segments]
    q = model.encode([question], return_dense=True, return_sparse=True)
    S = model.encode(seg_texts, return_dense=True, return_sparse=True)
    q_dense, q_lex = q["dense_vecs"][0], q["lexical_weights"][0]
    sem = S["dense_vecs"] @ q_dense
    lex = np.array([model.compute_lexical_matching_score(q_lex, lw)
                    for lw in S["lexical_weights"]])
    dual = lam * sem + (1 - lam) * lex
    thresh = np.percentile(dual, k)
    kept = [seg for seg, d in zip(segments, dual) if d >= thresh]
    return kept, dual


# Answerability Judgement

In [ ]:
from openai import OpenAI
from google.colab import userdata
client = OpenAI(api_key=userdata.get('DEEPINFRA_KEY'),base_url="https://api.deepinfra.com/v1/openai")
JUDGE_MODEL = "meta-llama/Llama-3.3-70B-Instruct-Turbo"

ANSWERABILITY_PROMPT = """You decide whether the Information is sufficient to answer the Question.
If sufficient, set answer to "answerable" and leave follow_up_question empty.
If insufficient, set answer to "unanswerable" and write ONE follow-up question naming the missing fact.
Return ONLY JSON of the form:
{"answer": "answerable" | "unanswerable", "follow_up_question": ""}"""
def judge_answerability(question, evidence, retries=6):
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=JUDGE_MODEL, temperature=0,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": ANSWERABILITY_PROMPT},
                    {"role": "user", "content": f"Question:\n{question}\n\nInformation:\n{evidence}"},
                ],
            )
            return json.loads(resp.choices[0].message.content)
        except Exception as e:
            msg = str(e).lower()
            if ("rate" in msg or "429" in msg) and attempt < retries - 1:
                time.sleep(2 ** attempt)      # exponential backoff: 1,2,4,8,16s
                continue
            # non-rate error or out of retries: stop iterating rather than crash
            return {"answer": "answerable", "follow_up_question": ""}



SecretNotFoundError: Secret DEEPINFRA_KEY does not exist.

# Missing Information Identification and Follow-up Question Generation

In [ ]:
def generate_followup_question(result):
  if result.get("answer") == "unanswerable":
      return result.get("follow_up_question")
  return None

# Iterative Evidence Accumulation

In [ ]:
def iterative_evidence_accumulation(question, segments, lam=0.6, k=90,
                                    max_iterations=5, use_judge=True, verbose=False):
    # ablation "no answerability judgment": single filter pass, no loop, no judge calls
    if not use_judge:
        filtered, _ = score_and_filter(question, segments, lam=lam, k=k)
        return filtered

    accumulated, query = [], question
    for i in range(max_iterations):
        filtered, _ = score_and_filter(query, segments, lam=lam, k=k)
        for s in filtered:
            if s not in accumulated:
                accumulated.append(s)
        evidence = "\n".join(x["text"] for x in accumulated)
        result = judge_answerability(question, evidence)
        if verbose:
            print(f"  iter {i+1}: query={query[:60]!r} | kept={len(accumulated)}"
                  f" | verdict={result.get('answer')}")
        if result.get("answer") == "answerable":
            break
        query = generate_followup_question(result) or question   # fall back if empty
    return accumulated

#Generate Answer

In [ ]:
_reader = None
def get_reader():
    global _reader
    if _reader is None:
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
        name = "NousResearch/Meta-Llama-3-8B-Instruct"
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
        tok = AutoTokenizer.from_pretrained(name)
        model = AutoModelForCausalLM.from_pretrained(name, quantization_config=bnb,
                                                     device_map="auto")
        _reader = (tok, model)
    return _reader

READER_PROMPT = ("Answer the question using ONLY the evidence. "
                 "Output ONLY the answer itself — a name, entity, number, or yes/no — "
                 "with no labels, prefixes, punctuation, or explanation. "
                 "If the evidence does not contain the answer, output exactly: unknown")

def generate_answer(question, evidence):
    tok, model = get_reader()
    msgs = [{"role": "system", "content": READER_PROMPT},
            {"role": "user", "content": f"Question:\n{question}\n\nEvidence:\n{evidence}"}]
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True).to(model.device)
    out = model.generate(**enc, max_new_tokens=64, do_sample=False,
                         pad_token_id=tok.eos_token_id)
    input_len = enc["input_ids"].shape[1]
    return tok.decode(out[0][input_len:], skip_special_tokens=True).strip()

# Evaluation

In [ ]:
def normalize_answer(s):
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def exact_match(pred, gold):
    return float(normalize_answer(pred) == normalize_answer(gold))

def f1_score(pred, gold):
    pt, gt = normalize_answer(pred).split(), normalize_answer(gold).split()
    if not pt or not gt:
        return float(pt == gt)
    common = collections.Counter(pt) & collections.Counter(gt)
    same = sum(common.values())
    if same == 0:
        return 0.0
    p, r = same / len(pt), same / len(gt)
    return 2 * p * r / (p + r)

def count_tokens(text):
    tok, _ = get_reader()
    return len(tok.encode(text))

# Run function

In [ ]:
def run_record(record, k, setting="itercomp", lam=0.6, max_iterations=5,
               use_judge=True, verbose=False):
    segs = decompose(record)
    if setting == "raw":            # feed everything (no compression) = "w/o filtering"
        kept = segs
    elif setting == "oracle":       # feed only the gold segments (upper bound)
        kept = [s for s in segs if s["is_supporting"]]
    else:                           # itercomp (and its ablations, via the knobs below)
        kept = iterative_evidence_accumulation(record["question"], segs, lam=lam, k=k,
                                               max_iterations=max_iterations,
                                               use_judge=use_judge, verbose=verbose)
    evidence = "\n".join(s["text"] for s in kept)
    pred = generate_answer(record["question"], evidence)
    raw_tokens = count_tokens("\n".join(s["text"] for s in segs))
    golds = [record["answer"]] + record.get("answer_aliases", [])   # MuSiQue: score vs aliases
    return {
        "question": record["question"],
        "gold": record["answer"],
        "pred": pred,
        "em": max(exact_match(pred, g) for g in golds),
        "f1": max(f1_score(pred, g) for g in golds),
        "ratio": count_tokens(evidence) / max(1, raw_tokens),
    }



def summarize(res):
    """Mean EM / F1 (percent) and ratio over a list of per-record result dicts."""
    return {"EM": 100 * np.mean([x["em"] for x in res]),
            "F1": 100 * np.mean([x["f1"] for x in res]),
            "ratio": float(np.mean([x["ratio"] for x in res]))}

def evaluate(recs, k, **kwargs):
    """Run one config; return (summary dict, list of per-record dicts)."""
    from tqdm.auto import tqdm
    res = [run_record(r, k, **kwargs) for r in tqdm(recs, leave=False)]
    return summarize(res), res

# Main

In [ ]:
K_by_dataset = {"hotpotqa": 90, "2wiki": 85, "musique": 90}
ABLATIONS = {
    "full":          dict(setting="itercomp", lam=0.6, max_iterations=5, use_judge=True),
    "no_iteration":  dict(setting="itercomp", lam=0.6, max_iterations=1, use_judge=True),
    "no_judge":      dict(setting="itercomp", lam=0.6, use_judge=False),
    "semantic_only": dict(setting="itercomp", lam=1.0, max_iterations=5, use_judge=True),
    "lexical_only":  dict(setting="itercomp", lam=0.0, max_iterations=5, use_judge=True),
    "no_filter":     dict(setting="raw"),      # w/o relevant evidence filtering
}

import os, csv
def _save(out_dir, summaries, predictions):
    os.makedirs(out_dir, exist_ok=True)
    if summaries:
        keys = sorted({k for row in summaries for k in row})
        with open(f"{out_dir}/summary.csv", "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=keys); w.writeheader(); w.writerows(summaries)
    with open(f"{out_dir}/predictions.jsonl", "w") as f:
        for p in predictions:
            f.write(json.dumps(p) + "\n")


def run_all(N=200, N_ablation=100, out_dir="itercomp_results"):
    summaries, predictions = [], []
    musique_main = None                      # (recs, res) reused for the hop table

    def add(table, dataset, config, summary, res, extra=None):
        summaries.append({"table": table, "dataset": dataset, "config": config,
                          "n": len(res), **(extra or {}), **summary})
        for x in res:
            predictions.append({"dataset": dataset, "config": config, **x})
        _save(out_dir, summaries, predictions)     # checkpoint

    # main results (Table 3): itercomp / raw / oracle on all three datasets, N each
    for name in ["hotpotqa", "musique"]:
        recs = get(name, n=N)
        for setting in ["itercomp", "raw", "oracle"]:
            print(f"[main] {name} / {setting}")
            s, res = evaluate(recs, K_by_dataset[name], setting=setting)
            add("main", name, setting, s, res)
            if name == "musique" and setting == "itercomp":
                musique_main = (recs, res)     # stash for hop analysis (no re-run)

    # ablations (Table 5) on MuSiQue, at the smaller N_ablation
    recs = get("musique", n=N_ablation)
    for variant, cfg in ABLATIONS.items():
        print(f"[ablation] {variant}  (n={N_ablation})")
        s, res = evaluate(recs, K_by_dataset["musique"], **cfg)
        add("ablation", "musique", variant, s, res)

    # hop-length analysis (Table 4): reuse the MuSiQue main IterCOMP run — zero extra calls
    recs, res = musique_main
    by_hop = {}
    for r, x in zip(recs, res):
        h = sum(p["is_supporting"] for p in r["paragraphs"])   # ~ hop count
        by_hop.setdefault(h, []).append(x)
    for h in sorted(by_hop):
        group = by_hop[h]
        summaries.append({"table": "hop", "dataset": "musique", "config": f"{h}hop",
                          "n": len(group), **summarize(group)})
    _save(out_dir, summaries, predictions)

    # final print
    for table in ["main", "ablation", "hop"]:
        print(f"\n=== {table} ===")
        for row in summaries:
            if row["table"] == table:
                print(f'{row["dataset"]:9} {row["config"]:14} n={row["n"]:<4} '
                      f'EM {row["EM"]:6.2f}  F1 {row["F1"]:6.2f}  ratio {row["ratio"]:.3f}')
    print(f"\nSaved: {out_dir}/summary_150.csv  and  {out_dir}/predictions_150.jsonl")
    return summaries, predictions



def show_trace(name="hotpotqa"):
    r = get(name, n=1)[0]
    segs = decompose(r)
    print("Q:", r["question"], "\nGold:", r["answer"], "\n")
    kept = iterative_evidence_accumulation(r["question"], segs,
                                           k=K_by_dataset[name], verbose=True)
    print("\nfinal answer:", generate_answer(r["question"], "\n".join(s["text"] for s in kept)))


if __name__ == "__main__":
    #run_all(N=3, N_ablation=3)
    #run_all(N=150, N_ablation=100,out_dir="/content/drive/MyDrive/itercomp_results")
    show_trace("hotpotqa")  # uncomment to print one iteration trace
